# Exercise 31.1 solution


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import rice_model_2008 as rice

## 31.1a: Steady-state Force-Calcium relations

This code is provided in the exercise — we run the model across a range of constant calcium concentrations and record the steady-state force.


In [ ]:
Cai_array = np.logspace(-2, 1, 50)
Fss = np.zeros_like(Cai_array)

t_span = (0, 500)
init_state = rice.init_state_values(SL=2.2)
force_index = rice.monitor_indices("active")

for i, Ca_level in enumerate(Cai_array):
    p = rice.init_parameter_values(
        start_time=2000, Ca_diastolic=Ca_level, SLmin=2.5, nperm=1
    )
    sol = solve_ivp(rice.rhs, t_span, init_state, args=(p,), method="BDF")
    final_state = sol.y[:, -1]
    m = rice.monitor(final_state, sol.t[-1], p)
    Fss[i] = m[force_index]

plt.figure(figsize=(7, 5))
plt.semilogx(Cai_array, Fss, color="dodgerblue", linewidth=2)
plt.ylabel("Normalized Force at Steady State")
plt.xlabel("Calcium Concentration (µM)")
plt.title("Steady-State Force-Calcium Relation")
plt.grid(alpha=0.3)
plt.show()

**Answer:** The curve resembles a **Hill function** (sigmoidal on a log scale). The steepness — characterised by the Hill coefficient $n_H$ — reflects the degree of **cooperativity** in the myofilament system. A Hill coefficient significantly greater than 1 (typically 3–7 for cardiac muscle) indicates that the binding of calcium to one site facilitates binding at neighbouring sites, producing the all-or-nothing-like activation that is essential for efficient cardiac contraction.


## 31.1b: The isometric twitch


In [ ]:
t_span = (0, 800)
t_eval = np.linspace(0, 800, 500)

p = rice.init_parameter_values(SLmin=2.5)
SL_values = [1.9, 2.1, 2.3]

plt.figure(figsize=(8, 5))

for SL in SL_values:
    init_state = rice.init_state_values(SL=SL)

    sol = solve_ivp(
        rice.rhs, t_span, init_state, args=(p,), method="BDF", t_eval=t_eval
    )

    force = np.zeros_like(sol.t)
    for i in range(len(sol.t)):
        m = rice.monitor(sol.y[:, i], sol.t[i], p)
        force[i] = m[force_index]

    plt.plot(sol.t, force, label=f"SL = {SL} µm", linewidth=2)

plt.xlabel("Time (ms)")
plt.ylabel("Active Force")
plt.title("Isometric Twitch at Different Sarcomere Lengths")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Observation:** Increasing the sarcomere length (from 1.9 to 2.3 µm) dramatically increases the peak twitch force, even though the identical calcium transient is used in all cases. This is the **Frank-Starling mechanism** at the myofilament level: stretching the sarcomere increases the calcium sensitivity and the number of available crossbridge binding sites, leading to greater force production.

The twitch duration also increases slightly with sarcomere length, reflecting the slower crossbridge cycling kinetics at longer lengths.
